# About this notebook
This notebook is for batching 2D analysis on multiple images at once.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from glob import glob

import pandas as pd

from iob_ia.utils import filter_labels
from iob_ia.utils.io_utils import (
    gen_out_path,
    read_image,
    read_labels,
    save_image_channel,
    save_labels,
)
from iob_ia.utils.measure import overlapping_labels
from iob_ia.utils.segment import segment_2d_sub_projections
from iob_ia.utils.workflows3d import check_images

## 1. Define variables and check images

In [3]:
# Define the path to the folder (use '/', and make sure that the path also ends with '/')
path_to_folder = "G:/20241120_IOB_Magdalena/20260715_memory_leak/images/"

# Image file type suffix
suffix = ".vsi"
# Channels that should be segmented (0-based indexing)
channels_to_segment = [1,2,3]
# In case metadata is missing (will be checked later...), specify the voxel size here
voxel_size = None  # or somthing like: (1.46, 0.65, 0.65)
# Range for substack projections before 2D segmentation
substack_size = 40.0

# workflow identifier, used for saving images (no need to change that)
wf = "IOB-2D-workflow_"

# Set thresholds separately per channel
cellprob = {1: -1.0, 2: -4.5, 3: -2.0}
flow = {1: 0.8, 2: 0.85, 3: 0.85}

# Filtering: a list of dictionaries (same order as channels_to_segment)
filters = [
    # For the first channel (GFP, 488C)
    {
        "projected_area": [42, 350],
        "intensity_mean": ["mean/4", float("inf")],
        #"intensity_mean": [250, float("inf")],
    },
    # For the second channel (Chx10, 568C)
    {
        "projected_area": [42, 350],
        "intensity_mean": [150, None],
        "intensity_max": [float("-inf"), 15000]
    },
    # For the third channel (Isl1, 640C)
    {
        "projected_area": [42, 350],
        "intensity_mean": [150, None],
        "intensity_max": [float("-inf"), 15000]
    }
]

*Info* - Filter dictionaires support dynamic values:

Instead of putting [min, max] absolute values for a certain measurement, you can define dynamic calculation based on the measurements itself.
Supported are:
- bimodal (same function as "estimate_threshold")
- mean
- median

Further you can adjust the dynamic thresholds by multiplication or division. E.g. by defining dictionaries as:
```
    {
        "area": ["median/4", "mean*5.5"],
        "intensity_mean": ["bimodal/2", float("inf")],
    }
```

## 2. Check images
Here we check if the images have correct and same metadata, as well as, whether the channels for segmentation are available in all images.


<font color='red'>WARNING: For testing purposes I 'manually' respecified to only use the first image (twice) for the analysis.</font>

<font color='red'>Please remove the 2nd code-line in the cell below.</font>

In [4]:
# check the image paths
image_paths = glob(path_to_folder + "*" + suffix)

for p in image_paths:
    print(p)
voxel_size = (5, 0.65, 0.65)
voxel_size = check_images(image_paths=image_paths, channels_to_segment=channels_to_segment, voxel_size=voxel_size)
print('Final voxel size:', voxel_size)

G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01.vsi
G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E3_01.vsi
G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E4_01.vsi
G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E5_01.vsi
G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E6_01.vsi
G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E7_01.vsi
G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E8_01.vsi
G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E9_01.vsi
G:/2024

100%|██████████████████████████████████████████████████████████████████████████████████| 60/60 [03:41<00:00,  3.69s/it]

Using manually defined voxel size.
All good!
Final voxel size: (5, 0.65, 0.65)


## 3. Segment images

In [5]:
if voxel_size is None:
    raise RuntimeError("The voxel size does not seem to be defined, probably the image check did not succeed...")
# Note do not keep images in memory
cp_mask_paths = []
image_channel_paths = []
image_channel_names = []
cur_count = 1
mod_voxel_size = None  # place holder for projected voxel size
for p in image_paths:
    mask_paths = []
    channel_paths = []
    # Read the image
    image = read_image(p)
    image_channel_names.append(image.channel_names)
    img = None  # but don't load the image data yet
    for ch in channels_to_segment:
        cp_mask_path = gen_out_path(p, wf + f'cp-mask_ch{ch}-{image.channel_names[ch]}')
        channel_path = gen_out_path(p, wf + f'image_ch{ch}-{image.channel_names[ch]}.ome')
        if os.path.exists(cp_mask_path):
            print('Already segmented:', p, 'Channel', ch, image.channel_names[ch])
            # Load the mask
            cp_mask = read_labels(cp_mask_path)
            # If the corresponding image channel does not exist, create it and check if the size matches the mask
            if not os.path.exists(channel_path):
                import numpy as np
                if img is None:
                    print("Loading image", p, "...")
                    img = image.get_data()
                n_slices = int(round(substack_size / voxel_size[0], 0))
                img_ch = []
                for z in range(0, img[ch].shape[0], n_slices):
                    substack = np.max(img[ch][z:z + n_slices, :, :], axis=0)
                    img_ch.append(substack)
                img_ch = np.asarray(img_ch)
            else:
                # Load the image channel
                image_channel = read_image(channel_path)
                img_ch = image_channel.get_data()
            # Check that the loaded image channel and mask have the same shape
            if cp_mask.shape != img_ch.shape:
                raise RuntimeError(
                    f'Mask shape ({cp_mask.shape}) and image channel shape ({img_ch.shape}) do not match. '
                    "Please delete mask and image channel images from previous runs:\n"
                    f'  - {cp_mask_path}\n  - {channel_path}\n  - and any other associated images...'
                )
            mask_paths.append(cp_mask_path)
            channel_paths.append(channel_path)
            # Remember the modified voxel size
            if mod_voxel_size is None:
                mod_voxel_size = (substack_size, voxel_size[1], voxel_size[2])
            # Save th image channel, if it does not exist
            if not os.path.exists(channel_path):
                save_image_channel(img_ch, channel_path, voxel_size=mod_voxel_size)
            del cp_mask, img_ch
        else:
            # Segment - Load the image data
            if img is None:
                print("Loading image", p, "...")
                img = image.get_data()
            cp_mask, img_ch, mod_voxel_size = segment_2d_sub_projections(
                img[ch],
                voxel_size=voxel_size,
                substack_size=substack_size,
                cellprob_threshold=cellprob[ch],
                flow_threshold=flow[ch]
            )
            # Save the label image
            save_labels(cp_mask, cp_mask_path)
            # Save the single (raw) channel
            save_image_channel(img_ch, channel_path, voxel_size=mod_voxel_size)
            # Add the paths to the lists
            mask_paths.append(cp_mask_path)
            channel_paths.append(channel_path)
            del cp_mask, img_ch
    del img
    del image
    cp_mask_paths.append(mask_paths)
    image_channel_paths.append(channel_paths)
    print(f'\n    --- Finished processing image {cur_count} out of {len(image_paths)}\n')
    cur_count += 1
print("Finished segmenting all images and channels.")

Already segmented: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01.vsi Channel 1 488C
Already segmented: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01.vsi Channel 2 561C
Already segmented: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01.vsi Channel 3 640C

    --- Finished processing image 1 out of 60

Already segmented: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E3_01.vsi Channel 1 488C
Already segmented: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E3_01.vsi Channel 2 561C
Already segmented: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E3_01.vsi Channel 3 640C



## 4. Filtering label masks

In [6]:
# Check that everythign is available
masks_ok = True
channels_ok = True
for i in range(len(cp_mask_paths)):
    if None in cp_mask_paths[i]:
        masks_ok = False
    if None in image_channel_paths[i]:
        channels_ok = False
if not masks_ok:
    raise RuntimeError(f'Cannot proceed!\nProblems with mask paths:\n{cp_mask_paths}')
if not channels_ok:
    raise RuntimeError(f'Cannot proceed!\nProblems with image channel paths:\n{image_channel_paths}')

# Create filtered masks, skip existing filtered masks
filtered_mask_paths = []
filtered_properties = []
cur_count = 1
for i in range(len(cp_mask_paths)):
    filtered_paths = []
    filtered_props = []
    for ch in range(len(cp_mask_paths[i])):
        cp_mask_path = cp_mask_paths[i][ch]
        channel_path = image_channel_paths[i][ch]
        filtered_path = cp_mask_path.replace('cp-mask_ch', "filtered-mask_ch")

        if os.path.exists(filtered_path):
            print(
                'Already filtered:',
                image_paths[i],
                'Channel',
                ch,
                image_channel_names[i][channels_to_segment[ch]]
            )

            filtered_paths.append(filtered_path)
            filtered_props.append(None)
            continue

        # Load the image an mask
        cp_mask = read_labels(cp_mask_path)
        channel = read_image(channel_path).get_data()
        # Measure properties
        properties = filter_labels.measure_label_props(cp_mask, channel, voxel_size=mod_voxel_size)
        del channel
        # Filter the mask and save it
        filtered_mask, updated_properties = filter_labels.filter_labels_by_property(cp_mask, props_table=properties, props=filters[ch])
        save_labels(filtered_mask, filtered_path)
        filtered_paths.append(filtered_path)
        filtered_props.append(updated_properties)
        del filtered_mask, cp_mask
    filtered_mask_paths.append(filtered_paths)
    filtered_properties.append(filtered_props)
    print(f'\n    --- Finished processing image {cur_count} out of {len(image_paths)}\n')
    cur_count += 1

Already filtered: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01.vsi Channel 0 488C
Already filtered: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01.vsi Channel 1 561C
Already filtered: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01.vsi Channel 2 640C

    --- Finished processing image 1 out of 60

Already filtered: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E3_01.vsi Channel 0 488C
Already filtered: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E3_01.vsi Channel 1 561C
Already filtered: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E3_01.vsi Channel 2 640C

    --

## 5. Create double and triple positive objects masks

In [7]:
# Count double positive objects
dp_mask_paths = []

for i in range(len(filtered_mask_paths)):
    cur_dp_paths = []

    if len(filtered_mask_paths[i]) != len(channels_to_segment):
        raise ValueError(
            "filtered_mask_paths[i] and channels_to_segment must have the same length."
        )

    if len(filtered_mask_paths[i]) < 2:
        print(f"Counting double positive objects requires at least 2 channels for image {image_paths[i]}.")
        dp_mask_paths.append(cur_dp_paths)
        continue

    for j in range(len(filtered_mask_paths[i])):
        for k in range(len(filtered_mask_paths[i])):
            if j == k:
                continue

            print(
                "--> Counting double positive objects for image:",
                image_paths[i],
                "channels:",
                image_channel_names[i][channels_to_segment[j]],
                "and",
                image_channel_names[i][channels_to_segment[k]]
            )

            # Load the masks
            mask_j = read_labels(filtered_mask_paths[i][j])
            mask_k = read_labels(filtered_mask_paths[i][k])

            # Create saving path
            dp_path = (
                f'mask_{image_channel_names[i][channels_to_segment[j]]}'
                f'-pos-{image_channel_names[i][channels_to_segment[k]]}-objects'
            )
            dp_path = gen_out_path(image_paths[i], wf + dp_path)

            # Count overlapping labels
            dp_mask = overlapping_labels(mask_j, mask_k, min_overlap=0.5)
            save_labels(dp_mask, dp_path)

            cur_dp_paths.append(dp_path)

            del mask_j, mask_k, dp_mask

    dp_mask_paths.append(cur_dp_paths)


# Count triple positive objects: GFP objects positive for 561 and 640
tp_mask_paths = []

for i in range(len(filtered_mask_paths)):
    cur_tp_paths = []

    if len(filtered_mask_paths[i]) != len(channels_to_segment):
        raise ValueError(
            "filtered_mask_paths[i] and channels_to_segment must have the same length."
        )

    if len(filtered_mask_paths[i]) < 3:
        print(f"Counting triple positive objects requires at least 3 channels for image {image_paths[i]}.")
        tp_mask_paths.append(cur_tp_paths)
        continue

    for j in range(len(filtered_mask_paths[i])):
        for k in range(len(filtered_mask_paths[i])):
            for L in range(len(filtered_mask_paths[i])):
                if j == k or j == L or k == L:
                    continue

                ch_j = image_channel_names[i][channels_to_segment[j]]
                ch_k = image_channel_names[i][channels_to_segment[k]]
                ch_l = image_channel_names[i][channels_to_segment[L]]

                # Only GFP objects positive for 561 and 640
                if "488" not in ch_j:
                    continue
                if not ("561" in ch_k and "640" in ch_l):
                    continue

                print(
                    "--> Counting triple positive objects for image:",
                    image_paths[i],
                    "channels:",
                    ch_j,
                    "and",
                    ch_k,
                    "and",
                    ch_l
                )

                # Load the masks
                mask_j = read_labels(filtered_mask_paths[i][j])
                mask_k = read_labels(filtered_mask_paths[i][k])
                mask_l = read_labels(filtered_mask_paths[i][L])

                # Create saving path
                tp_path = (
                    f'mask_{ch_j}'
                    f'-pos-{ch_k}'
                    f'-pos-{ch_l}-objects'
                )
                tp_path = gen_out_path(image_paths[i], wf + tp_path)

                # Count overlapping labels
                tmp_mask = overlapping_labels(mask_j, mask_k, min_overlap=0.5)
                tp_mask = overlapping_labels(tmp_mask, mask_l, min_overlap=0.5)

                save_labels(tp_mask, tp_path)

                cur_tp_paths.append(tp_path)

                del mask_j, mask_k, mask_l, tmp_mask, tp_mask

    tp_mask_paths.append(cur_tp_paths)

--> Counting double positive objects for image: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01.vsi channels: 488C and 561C
Saved labels to: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01_IOB-2D-workflow_mask_488C-pos-561C-objects.tif
--> Counting double positive objects for image: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01.vsi channels: 488C and 640C
Saved labels to: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01_IOB-2D-workflow_mask_488C-pos-640C-objects.tif
--> Counting double positive objects for image: G:/20241120_IOB_Magdalena/20260715_memory_leak/images\20260422_LUe033_bipolar cell serotypes_10x_plate1_2-2 1E+10_E10_01.vsi channels: 561C and 488C
Saved labels to: G:/20241120_IOB_Magdal

## 6. Count all objects and create result table

In [8]:
result_table = None

for i_img in range(len(cp_mask_paths)):
    if len(cp_mask_paths[i_img]) != len(channels_to_segment):
        raise ValueError("cp_mask_paths[i_img] and channels_to_segment must have the same length.")
    if len(filtered_mask_paths[i_img]) != len(channels_to_segment):
        raise ValueError("filtered_mask_paths[i_img] and channels_to_segment must have the same length.")

    # Create result table
    if result_table is None:
        result_table = {
            "Image-Path": image_paths,
            "Image-Name": [],
        }

        for ch in channels_to_segment:
            result_table[f'cp-count-ch{ch}'] = []
        for ch in channels_to_segment:
            result_table[f'object-count-ch{ch}'] = []

        for i_ch in range(len(channels_to_segment)):
            for j_ch in range(len(channels_to_segment)):
                if i_ch == j_ch:
                    continue
                result_table[
                    f'ch{channels_to_segment[i_ch]}pos-ch{channels_to_segment[j_ch]}objects'
                ] = []

        # Triple positive objects
        result_table["ch2pos-ch3pos-ch1objects"] = []

    # Image file names
    result_table["Image-Name"].append(
        os.path.basename(image_paths[i_img])
    )

    print(f'Counting image {i_img + 1} out of {len(cp_mask_paths)} images...')

    # Mask object counts
    for i_ch in range(len(cp_mask_paths[i_img])):
        # Load masks
        cp_mask = read_labels(cp_mask_paths[i_img][i_ch])
        filtered_mask = read_labels(filtered_mask_paths[i_img][i_ch])

        # Count objects
        result_table[f'cp-count-ch{channels_to_segment[i_ch]}'].append(
            filter_labels.count_unique_labels(cp_mask)
        )
        result_table[f'object-count-ch{channels_to_segment[i_ch]}'].append(
            filter_labels.count_unique_labels(filtered_mask)
        )

        del cp_mask, filtered_mask

    # Double positive counts
    if dp_mask_paths[i_img]:
        dp_idx = 0
        for i_ch in range(len(channels_to_segment)):
            for j_ch in range(len(channels_to_segment)):
                if i_ch == j_ch:
                    continue

                dp_mask = read_labels(dp_mask_paths[i_img][dp_idx])
                result_table[
                    f'ch{channels_to_segment[i_ch]}pos-ch{channels_to_segment[j_ch]}objects'
                ].append(
                    filter_labels.count_unique_labels(dp_mask)
                )
                del dp_mask
                dp_idx += 1
    else:
        for i_ch in range(len(channels_to_segment)):
            for j_ch in range(len(channels_to_segment)):
                if i_ch == j_ch:
                    continue
                result_table[
                    f'ch{channels_to_segment[i_ch]}pos-ch{channels_to_segment[j_ch]}objects'
                ].append(None)

    # Triple positive counts
    if tp_mask_paths[i_img]:
        tp_mask = read_labels(tp_mask_paths[i_img][0])

        result_table["ch2pos-ch3pos-ch1objects"].append(
            filter_labels.count_unique_labels(tp_mask)
        )

        del tp_mask
    else:
        result_table["ch2pos-ch3pos-ch1objects"].append(None)

result_table = pd.DataFrame.from_dict(result_table)

# Save table
rt_path = path_to_folder + wf + 'batch_result_table.csv'
pd.DataFrame.to_csv(result_table, rt_path)
print("Saved result table to:", rt_path)

result_table

Counting image 1 out of 60 images...
Counting image 2 out of 60 images...
Counting image 3 out of 60 images...
Counting image 4 out of 60 images...
Counting image 5 out of 60 images...
Counting image 6 out of 60 images...
Counting image 7 out of 60 images...
Counting image 8 out of 60 images...
Counting image 9 out of 60 images...
Counting image 10 out of 60 images...
Counting image 11 out of 60 images...
Counting image 12 out of 60 images...
Counting image 13 out of 60 images...
Counting image 14 out of 60 images...
Counting image 15 out of 60 images...
Counting image 16 out of 60 images...
Counting image 17 out of 60 images...
Counting image 18 out of 60 images...
Counting image 19 out of 60 images...
Counting image 20 out of 60 images...
Counting image 21 out of 60 images...
Counting image 22 out of 60 images...
Counting image 23 out of 60 images...
Counting image 24 out of 60 images...
Counting image 25 out of 60 images...
Counting image 26 out of 60 images...
Counting image 27 out

,Image-Path,Image-Name,cp-count-ch1,cp-count-ch2,cp-count-ch3,object-count-ch1,object-count-ch2,object-count-ch3,ch1pos-ch2objects,ch1pos-ch3objects,ch2pos-ch1objects,ch2pos-ch3objects,ch3pos-ch1objects,ch3pos-ch2objects,ch2pos-ch3pos-ch1objects
0,G:/20241120_IOB_Magdalena/20260715_memory_leak...,20260422_LUe033_bipolar cell serotypes_10x_pla...,2444,8657,7273,1894,8475,6924,1081,783,583,4846,579,5448,696
1,G:/20241120_IOB_Magdalena/20260715_memory_leak...,20260422_LUe033_bipolar cell serotypes_10x_pla...,851,3427,3899,320,2918,3489,92,68,26,1734,45,1941,41
2,G:/20241120_IOB_Magdalena/20260715_memory_leak...,20260422_LUe033_bipolar cell serotypes_10x_pla...,3123,5733,6706,2461,4826,6222,758,519,289,3118,358,3482,360
3,G:/20241120_IOB_Magdalena/20260715_memory_leak...,20260422_LUe033_bipolar cell serotypes_10x_pla...,4907,4211,3655,3885,3398,3389,726,515,262,1524,361,1687,258
4,G:/20241120_IOB_Magdalena/20260715_memory_leak...,20260422_LUe033_bipolar cell serotypes_10x_pla...,5772,769,1933,4985,0,3,0,2,0,0,1,0,0
5,G:/20241120_IOB_Magdalena/20260715_memory_leak...,20260422_LUe033_bipolar cell serotypes_10x_pla...,3674,4398,4224,2397,3968,3679,655,385,200,1838,214,1996,221
6,G:/20241120_IOB_Magdalena/20260715_memory_leak...,20260422_LUe033_bipolar cell serotypes_10x_pla...,1915,3001,2969,1337,2755,2818,446,276,156,1570,177,1774,213
7,G:/20241120_IOB_Magdalena/20260715_memory_leak...,20260422_LUe033_bipolar cell serotypes_10x_pla...,4053,6236,5120,2474,5906,4161,620,283,223,2504,173,2798,204
8,G:/20241120_IOB_Magdalena/20260715_memory_leak...,20260422_LUe033_bipolar cell serotypes_10x_pla...,3437,7573,8376,2760,6769,7869,1140,770,639,3961,590,4396,608
9,G:/20241120_IOB_Magdalena/20260715_memory_leak...,20260422_LUe033_bipolar cell serotypes_10x_pla...,8931,6772,8528,7684,6283,7883,1517,1638,786,4795,1377,5307,1057
